In [21]:
import os

DATAPATH = "/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs"

# Map each dataset name -> the exact adjacency-list file to use.
# Paths may be absolute or just a filename (resolved against DATAPATH below).
ADJ_LISTS = {
    # "mnist":         "adj-list-mnist-euclidean.txt",
    # "fashion_mnist": "adj-list-fashion_mnist-euclidean.txt",
    # "glove25":       "adj-list-glove25-euclidean.txt",
    "coco_i2i":      "adj-list-coco_i2i-euclidean.txt",
}

# beam_search.py knobs
BEAM_WIDTHS  = [1, 2, 4, 8, 16, 32, 64, 100, 128, 256]
MIN_COVERAGE = 99.5
STEP_SIZE    = 0.1
TESTS        = 100 

print("Files in DATAPATH:")
for f in sorted(os.listdir(DATAPATH)):
    print(" ", f)

Files in DATAPATH:
  adj-list-coco_i2i-euclidean.txt
  adj-list-coco_t2i-euclidean.txt
  adj-list-fashion_mnist-euclidean.txt
  adj-list-glove25-euclidean.txt
  adj-list-mnist-euclidean.txt
  coco_i2i-512-angular.hdf5
  fashion_mnist-784-euclidean.hdf5
  glove25-25-angular.hdf5
  mnist-784-euclidean.hdf5
  set-cover-adj-list-fashion_mnist-euclidean.txt
  set-cover-adj-list-mnist-euclidean.txt


In [22]:
import glob
import subprocess
import sys

def find_hdf5(name):
    """Locate the .hdf5 file for a dataset name (prefix before the first '-')."""
    matches = sorted(glob.glob(os.path.join(DATAPATH, f"{name}-*.hdf5")))
    if not matches:
        raise FileNotFoundError(f"No hdf5 for dataset '{name}' in {DATAPATH}")
    return matches[0]

def resolve(path):
    """Allow bare filenames in ADJ_LISTS, resolved against DATAPATH."""
    return path if os.path.isabs(path) else os.path.join(DATAPATH, path)

beam_search_dir = os.path.dirname(os.path.abspath("beam_search.py"))
beam_search_py  = os.path.join(beam_search_dir, "beam_search.py")
if not os.path.exists(beam_search_py):
    beam_search_py  = "beam_search.py"          # fall back to cwd
    beam_search_dir = os.getcwd()

# Persist results to ../beam_search_long_sc.csv (repo root, one level up from
# beam_search/). beam_search.py treats --save_path as a directory and joins
# --out_csv onto it; it writes LONG (df_long) format -- one row per
# (dataset, beam_width, k, coverage) -- and upserts on that key.
SAVE_DIR = os.path.abspath(os.path.join(beam_search_dir, ".."))
OUT_CSV  = "beam_search_test.csv"
print(f"Results -> {os.path.join(SAVE_DIR, OUT_CSV)}\n")

def run_inherit(cmd):
    """
    Run cmd letting it write DIRECTLY to the notebook's stdout/stderr (no pipes),
    so tqdm's in-place '\\r' bar redraws work exactly as they did with the
    original subprocess.run. On KeyboardInterrupt the child is terminated cleanly.
    Returns the exit code.
    """
    proc = subprocess.Popen(cmd)            # inherits stdout/stderr — no capture
    try:
        proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    return proc.returncode

for name, adj_list in ADJ_LISTS.items():
    hdf5     = find_hdf5(name)
    adj_list = resolve(adj_list)
    if not os.path.exists(adj_list):
        raise FileNotFoundError(f"adj-list for '{name}' not found: {adj_list}")

    cmd = [
        sys.executable, beam_search_py,
        "--dataset",      hdf5,
        "--adj_list",     adj_list,
        "--save_path",    SAVE_DIR,
        "--out_csv",      OUT_CSV,
        "--beam_widths",  *[str(b) for b in BEAM_WIDTHS],
        "--min_coverage", str(MIN_COVERAGE),
        "--step_size",    str(STEP_SIZE),
        "--tests",        str(TESTS),
    ]

    print("=" * 80)
    print(f"Running beam search: {name}")
    print(f"  dataset      : {hdf5}")
    print(f"  adj_list     : {adj_list}")
    print(f"  beam_widths  : {BEAM_WIDTHS}")
    print(f"  min_coverage : {MIN_COVERAGE}   step_size : {STEP_SIZE}   tests : {TESTS}")
    print(f"  -> {os.path.join(SAVE_DIR, OUT_CSV)}")
    print("=" * 80, flush=True)

    rc = run_inherit(cmd)
    if rc != 0:
        print(f"!! {name} exited with code {rc}", flush=True)
    else:
        print(f"✓ {name} done", flush=True)


Results -> /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/navigable-graphs/beam_search_test.csv

Running beam search: coco_i2i
  dataset      : /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/coco_i2i-512-angular.hdf5
  adj_list     : /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/adj-list-coco_i2i-euclidean.txt
  beam_widths  : [1, 2, 4, 8, 16, 32, 64, 100, 128, 256]
  min_coverage : 99.5   step_size : 0.1   tests : 100
  -> /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/navigable-graphs/beam_search_test.csv
Loading {'name': 'coco_i2i', 'filepath': '/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/coco_i2i-512-angular.hdf5', 'adj_list': '/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/adj-list-coco_i2i-euclidean.txt'}
Building CSR graphs...


113287it [00:22, 5074.49it/s]


Avg out-degrees
--------------
100.0% navigable: 28.59
99.9% navigable: 19.02
99.8% navigable: 16.66
99.7% navigable: 15.27
99.6% navigable: 14.29
--------------

Searching test queries (n=100, beam_widths=[1, 2, 4, 8, 16, 32, 64, 100, 128, 256])...


100%|████████████████████████████████████████| 100/100 [00:24<00:00,  4.01it/s]



=== Test queries  beam_width=1  recall@1 ===
            metric  G_1.0  G_0.9990000000000001  G_0.9980000000000001  G_0.9970000000000002  G_0.9960000000000002
        avg recall   0.45                  0.48                  0.49                  0.40                  0.35
    avg nodes seen 203.71                166.27                155.94                143.71                135.90
avg nodes expanded   7.36                  8.43                  8.61                  8.48                  8.61

=== Test queries  beam_width=1  recall@10 ===
            metric   G_1.0  G_0.9990000000000001  G_0.9980000000000001  G_0.9970000000000002  G_0.9960000000000002
        avg recall   0.087                  0.08                 0.075                 0.072                 0.069
    avg nodes seen 203.710                166.27               155.940               143.710               135.900
avg nodes expanded   7.360                  8.43                 8.610                 8.480              

In [ ]:
import pandas as pd

# Both files are now LONG (df_long) format: one row per
# (dataset, beam_width, k, coverage). No melt needed.
df_sc = pd.read_csv("../beam_search_long_sc.csv")   # set-cover
df    = pd.read_csv("../beam_search_long.csv")       # robust-prune

df_long = df_sc.copy()
df_long['coverage'] = df_long['coverage'].astype(float)
sorted(df_long['coverage'].unique(), reverse=True)


FileNotFoundError: [Errno 2] No such file or directory: '../beam_search_long.csv'

In [ ]:
import matplotlib.pyplot as plt

def prep(frame):
    """Long beam-search summary (already one row per (dataset, bw, k, coverage)):
    coerce coverage to float and derive recall."""
    out = frame.copy()
    out['coverage'] = out['coverage'].astype(float)
    out['recall'] = out['train_relevant'] / out['k']
    return out

rp_long = prep(df)      # robust-prune
sc_long = prep(df_sc)   # set-cover

# Exclude full coverage; show only the sparser (almost-navigable) graphs.
rp_long = rp_long[rp_long['coverage'] < 1.0]
sc_long = sc_long[sc_long['coverage'] < 1.0]

# Scatter recall vs distance computations across the coverage < 1.0 levels.
# Rows = datasets present in both; cols = k. Each point is one (coverage, beam_width).
datasets = sorted(set(rp_long['dataset']) & set(sc_long['dataset']))
k_values = sorted(set(rp_long['k']) | set(sc_long['k']))
n_ds, n_k = len(datasets), len(k_values)

fig, axes = plt.subplots(n_ds, n_k, figsize=(5 * n_k, 4 * n_ds), squeeze=False)

for ri, dataset in enumerate(datasets):
    for ci, k in enumerate(k_values):
        ax = axes[ri, ci]
        for long, color, label in [(rp_long, 'tab:blue',  'robust-prune'),
                                    (sc_long, 'tab:orange', 'set-cover')]:
            sub = long[(long['dataset'] == dataset) & (long['k'] == k)]
            if sub.empty:
                continue
            ax.scatter(sub['train_seen'], sub['recall'],
                       s=12, alpha=0.8, color=color, label=label, edgecolors='none')

        ax.set_xlabel('Distance Computations')
        ax.set_ylabel(f'Recall@{k}')
        ax.set_title(f'{dataset}  |  Recall@{k}  (coverage < 100%)')
        ax.grid(alpha=0.3)
        if ri == 0 and ci == 0:
            ax.legend()

fig.tight_layout()
plt.show()
